# pytest fixtures — why your artifact test returns 409

**What this teaches:** How pytest fixtures work, why artifact tests need a specific setup sequence, and how to express that sequence as a reusable fixture.

**Why it matters:** You need this to write `test_artifacts.py` in Lab 01. But the pattern transfers to any test suite for a stateful API.

**Structure:** Encounter → trace → fix → concept → exercise. Work in order.

In [ ]:
# Setup check — run this first.
# Run from the deborgen repo root with: jupyter notebook
# If you get ModuleNotFoundError, activate the project environment:
#   uv run jupyter notebook

from fastapi.testclient import TestClient
from deborgen.coordinator.app import create_app

print("imports OK")

---
## Section 1: Encounter

You want to test `POST /jobs/{id}/artifacts`. The endpoint records a download URL on a job.

Here's the most natural first attempt. Run it.

In [ ]:
# Naive first attempt at testing the artifact endpoint.

app = create_app(db_path=":memory:")
client = TestClient(app)

# Submit a job
resp = client.post("/jobs", json={"command": "echo hello"})
job_id = resp.json()["id"]
print("job created:", job_id)

# Try to record an artifact URL
resp = client.post(f"/jobs/{job_id}/artifacts", json={
    "node_id": "my-node",
    "lease_token": "some-token",
    "url": "https://example.com/artifacts.zip",
})
print("status:", resp.status_code)
print("body:", resp.json())

409. Not 200.

The detail says `"job has no active lease"`. What does that mean, and why does the artifact endpoint care?

---
## Section 2: Trace — what does `assert_job_lease` check?

Every route that modifies a running job — logs, artifacts, finish — starts by calling `assert_job_lease`. Here's the full function from `src/deborgen/coordinator/app.py`:

```python
def assert_job_lease(self, job_id: str, node_id: str, lease_token: str) -> None:
    job_pk = parse_job_pk(job_id)
    with self._lock:
        row = self._get_job_row(job_pk)
        if row is None:
            raise HTTPException(status_code=404, detail="job not found")
        lease = self._conn.execute(
            "SELECT node_id, lease_token, lease_expires_at FROM leases WHERE job_id = ?",
            (job_pk,),
        ).fetchone()
        if lease is None:
            raise HTTPException(status_code=409, detail="job has no active lease")
        ...
        if lease_node_id != node_id or lease_token_db != lease_token:
            raise HTTPException(status_code=409, detail="job is owned by a different worker")
        if utcnow() > lease_expires_at:
            raise HTTPException(status_code=409, detail="lease has expired")
```

It checks three things:
1. Does a lease row exist for this job?
2. Does the `node_id` + `lease_token` match what's in that row?
3. Has the lease expired?

A freshly submitted job has no lease row at all. That's why we hit check #1 immediately.

**Where does a lease come from?** Claiming a job. When a worker calls `GET /jobs/next`, the coordinator atomically sets the job status to `running` and writes a lease row with a generated token. That token is what the worker uses for every subsequent call.

---
## Section 3: Trace — the state machine

Run this to see the job status after each step.

In [ ]:
# Watch job status change as we move through the required steps.

app = create_app(db_path=":memory:")
client = TestClient(app)

# Step 1: submit
resp = client.post("/jobs", json={"command": "echo hello"})
job = resp.json()
job_id = job["id"]
print("after submit:   ", job["status"])  # queued

# Step 2: heartbeat a node (required before claiming)
client.post("/nodes/my-node/heartbeat", json={"labels": {}})

# Step 3: claim the job
resp = client.get("/jobs/next?node_id=my-node")
assignment = resp.json()
print("after claim:    ", assignment["job"]["status"])  # running
print("lease_token:    ", assignment["lease_token"])

# Now the artifact endpoint has what it needs
lease_token = assignment["lease_token"]
resp = client.post(f"/jobs/{job_id}/artifacts", json={
    "node_id": "my-node",
    "lease_token": lease_token,
    "url": "https://example.com/artifacts.zip",
})
print("artifact POST:  ", resp.status_code)  # 200

Three steps, not one: **submit → heartbeat → claim**. Only after claiming does a lease row exist, and only then do artifact/log/finish endpoints work.

The lease token ties everything together. It's proof of ownership — the coordinator issued it, and only the holding worker knows it.

---
## Section 4: The problem with inline setup

You need this three-step sequence in every artifact test. Here's what happens if you write it inline:

In [ ]:
# What test_artifacts.py looks like WITHOUT fixtures.
# This is real code — notice what's the same in every test.

def test_artifact_is_recorded_on_job():
    app = create_app(db_path=":memory:")
    client = TestClient(app)
    resp = client.post("/jobs", json={"command": "echo hello"})
    job_id = resp.json()["id"]
    client.post("/nodes/my-node/heartbeat", json={"labels": {}})
    assignment = client.get("/jobs/next?node_id=my-node").json()
    lease_token = assignment["lease_token"]
    # ^^^ 6 lines of setup — now the actual test:
    client.post(f"/jobs/{job_id}/artifacts", json={
        "node_id": "my-node", "lease_token": lease_token,
        "url": "https://example.com/a.zip",
    })
    job = client.get(f"/jobs/{job_id}").json()
    assert "https://example.com/a.zip" in job["artifact_urls"]

def test_artifact_survives_finish():
    app = create_app(db_path=":memory:")
    client = TestClient(app)
    resp = client.post("/jobs", json={"command": "echo hello"})
    job_id = resp.json()["id"]
    client.post("/nodes/my-node/heartbeat", json={"labels": {}})
    assignment = client.get("/jobs/next?node_id=my-node").json()
    lease_token = assignment["lease_token"]
    # ^^^ identical 6 lines again
    client.post(f"/jobs/{job_id}/artifacts", json={
        "node_id": "my-node", "lease_token": lease_token,
        "url": "https://example.com/a.zip",
    })
    client.post(f"/jobs/{job_id}/finish", json={
        "node_id": "my-node", "lease_token": lease_token, "exit_code": 0,
    })
    job = client.get(f"/jobs/{job_id}").json()
    assert "https://example.com/a.zip" in job["artifact_urls"]

# Run both to prove they work — then we'll fix the duplication
test_artifact_is_recorded_on_job()
test_artifact_survives_finish()
print("both pass")

Tests pass, but the setup is duplicated. Six more tests means six more copies of those six lines. And every copy is a place where you might write a slightly different node name, or forget the heartbeat, or get a stale `job_id`.

**Pause.** Before reading Section 5: what would a `running_job` fixture look like? What should it return so a test can use both `client` and `lease_token`? Write your answer in the cell below.

In [ ]:
# YOUR ATTEMPT: sketch the running_job fixture.
# What does it need to set up? What does it return?

import pytest

@pytest.fixture
def running_job(client):   # <-- accepts client as a fixture
    # YOUR CODE HERE
    ...

---
## Section 5: Fixtures — the concept

A pytest fixture is a function decorated with `@pytest.fixture`. When a test declares it as a parameter, pytest calls the fixture and passes the result in.

The key insight: **fixtures can depend on other fixtures**. `running_job` can accept `client` as a parameter — pytest sees that dependency and resolves it automatically.

In [ ]:
# Fixtures in isolation — no pytest runner needed, just the concept.

# Fixture 1: a fresh in-memory app client.
# In a real test file this would have @pytest.fixture.
def make_client():
    return TestClient(create_app(db_path=":memory:"))

# Fixture 2: depends on client, returns everything a test needs.
def make_running_job(client):
    resp = client.post("/jobs", json={"command": "echo hello"})
    job_id = resp.json()["id"]
    client.post("/nodes/test-node/heartbeat", json={"labels": {}})
    assignment = client.get("/jobs/next?node_id=test-node").json()
    return {
        "job_id": job_id,
        "node_id": "test-node",
        "lease_token": assignment["lease_token"],
    }

# A test that uses both:
def test_artifact_is_recorded(client, running_job):
    client.post(f"/jobs/{running_job['job_id']}/artifacts", json={
        "node_id": running_job["node_id"],
        "lease_token": running_job["lease_token"],
        "url": "https://example.com/a.zip",
    })
    job = client.get(f"/jobs/{running_job['job_id']}").json()
    assert "https://example.com/a.zip" in job["artifact_urls"]

# Simulate what pytest does:
_client = make_client()
_running_job = make_running_job(_client)
test_artifact_is_recorded(_client, _running_job)
print("pass — same client instance shared between fixtures")

The critical detail: pytest passes **the same `client` instance** into `running_job` and into the test. That's why the job submitted inside `running_job` is visible when the test calls `GET /jobs/{id}` — they're talking to the same in-memory database.

If each fixture created its own `TestClient(create_app(...))`, they'd have separate databases and the job would be invisible to the test.

Here's what this looks like as real pytest code:

In [ ]:
# The real fixture code — exactly what goes in conftest.py or test_artifacts.py.

import pytest
from fastapi.testclient import TestClient
from deborgen.coordinator.app import create_app

@pytest.fixture
def client() -> TestClient:
    return TestClient(create_app(db_path=":memory:"))

@pytest.fixture
def running_job(client: TestClient) -> dict:
    resp = client.post("/jobs", json={"command": "echo hello"})
    job_id = resp.json()["id"]
    client.post("/nodes/test-node/heartbeat", json={"labels": {}})
    assignment = client.get("/jobs/next?node_id=test-node").json()
    return {
        "job_id": job_id,
        "node_id": "test-node",
        "lease_token": assignment["lease_token"],
    }

# pytest wires these together automatically because running_job declares
# `client` as a parameter — same name, same fixture.

print("This is what goes in test_artifacts.py.")

---
## Section 6: Verify — the wrong-token case

The lab requires testing that a wrong lease token returns 409. Here's the contrast:

Before fixtures: you'd repeat the full setup just to get a `job_id` and then call with a bad token.
After: `running_job` gives you a valid setup, and you just pass the wrong token.

In [ ]:
# Simulate: test_artifact_wrong_token_returns_409

app = create_app(db_path=":memory:")
_client = TestClient(app)

# running_job fixture
resp = _client.post("/jobs", json={"command": "echo hello"})
job_id = resp.json()["id"]
_client.post("/nodes/test-node/heartbeat", json={"labels": {}})
assignment = _client.get("/jobs/next?node_id=test-node").json()
_running_job = {"job_id": job_id, "node_id": "test-node", "lease_token": assignment["lease_token"]}

# The test itself is one line — wrong token:
resp = _client.post(f"/jobs/{_running_job['job_id']}/artifacts", json={
    "node_id": _running_job["node_id"],
    "lease_token": "wrong-token",
    "url": "https://example.com/a.zip",
})
print("status:", resp.status_code)   # 409
print("detail:", resp.json()["detail"])

---
## Section 7: Exercise

You have the pattern. Now write three real tests for `test_artifacts.py`.

The cell below is the skeleton. Fill in each test body.

**Success condition for each:**
- `test_duplicate_url_appears_once`: `artifact_urls` has length 1, not 3
- `test_artifact_survives_finish`: `artifact_urls` is non-empty after `POST /jobs/{id}/finish`
- `test_presign_without_s3_returns_500`: status is 500, detail contains `"not configured"`

Don't open the hints until you've tried.

In [ ]:
# Run these in a real pytest session: uv run pytest test_artifacts_exercise.py -v
# Or prototype the logic here first — your choice.

import pytest
from fastapi.testclient import TestClient
from deborgen.coordinator.app import create_app

@pytest.fixture
def client() -> TestClient:
    return TestClient(create_app(db_path=":memory:"))

@pytest.fixture
def running_job(client: TestClient) -> dict:
    resp = client.post("/jobs", json={"command": "echo hello"})
    job_id = resp.json()["id"]
    client.post("/nodes/test-node/heartbeat", json={"labels": {}})
    assignment = client.get("/jobs/next?node_id=test-node").json()
    return {
        "job_id": job_id,
        "node_id": "test-node",
        "lease_token": assignment["lease_token"],
    }

def test_duplicate_url_appears_once(client: TestClient, running_job: dict) -> None:
    # Post the same URL three times, expect it to appear exactly once in artifact_urls
    ...

def test_artifact_survives_finish(client: TestClient, running_job: dict) -> None:
    # Record a URL, finish the job, check the URL is still there
    ...

def test_presign_without_s3_returns_500(client: TestClient, running_job: dict, monkeypatch: pytest.MonkeyPatch) -> None:
    # Ensure S3 env vars are absent, call presign, expect 500 with "not configured"
    # Hint: monkeypatch.delenv("S3_ENDPOINT_URL", raising=False)
    ...

<details>
<summary>Hint: test_duplicate_url_appears_once</summary>

Call `POST /jobs/{id}/artifacts` three times with the same URL. Then call `GET /jobs/{id}` and check `len(job["artifact_urls"]) == 1`. The deduplication lives in `record_artifact` — your test just proves it works.
</details>

<details>
<summary>Hint: test_artifact_survives_finish</summary>

Record a URL, then call `POST /jobs/{id}/finish` with `exit_code: 0`. Then `GET /jobs/{id}` and check that the URL is still in `artifact_urls`. The finish endpoint doesn't touch artifact_urls — your test proves that assumption.
</details>

<details>
<summary>Hint: test_presign_without_s3_returns_500</summary>

```python
for var in ["S3_ENDPOINT_URL", "S3_ACCESS_KEY_ID", "S3_SECRET_ACCESS_KEY", "S3_BUCKET_NAME"]:
    monkeypatch.delenv(var, raising=False)

resp = client.post(f"/jobs/{running_job['job_id']}/artifacts/presign", json={
    "node_id": running_job["node_id"],
    "lease_token": running_job["lease_token"],
    "filename": "artifacts.zip",
})
assert resp.status_code == 500
assert "not configured" in resp.json()["detail"]
```
</details>

---
## Summary

The pattern — applied to any stateful API test suite:

```python
# 1. Fresh database per test — isolation
@pytest.fixture
def client() -> TestClient:
    return TestClient(create_app(db_path=":memory:"))

# 2. Reusable state — built on top of client
@pytest.fixture
def running_job(client: TestClient) -> dict:
    # submit → heartbeat → claim
    # return the ids and tokens tests need
    ...

# 3. Tests declare what they need — pytest wires it
def test_something(client: TestClient, running_job: dict) -> None:
    # setup is already done — write only the assertion
    ...
```

The three things to remember:
- `:memory:` gives each test a fresh database with no shared state
- `running_job` depends on `client` — pytest passes the **same instance** to both
- The artifact endpoint requires a real lease: submit → heartbeat → claim, in that order

After this, you can write the remaining Lab 01 tests: wrong token, presign happy path (with mocked boto3), and the duplicate URL case.